# Hateful Memes V2 Training in Google Colab

This notebook keeps your existing `src/train_dynamic_v2.py` and `src/train_static_v2.py` code, but swaps the file paths and dataset download flow so the project can run inside Colab.

Before you run it:
1. Upload this repo to Colab or mount a Google Drive folder that contains it.
2. Set `REPO_DIR` in the next cells.
3. Choose `RUN_MODE = "dynamic_v2"` or `"static_v2"`.


In [ ]:
!pip install -q transformers huggingface_hub datasets pandas pyarrow scikit-learn pillow matplotlib

In [ ]:
from pathlib import Path
import os

USE_DRIVE = False
DRIVE_REPO_DIR = "/content/drive/MyDrive/Hateful memes static copy"
LOCAL_REPO_DIR = "/content/Hateful memes static copy"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path(DRIVE_REPO_DIR)
else:
    REPO_DIR = Path(LOCAL_REPO_DIR)

if not REPO_DIR.exists():
    raise FileNotFoundError(
        f"Set REPO_DIR to the uploaded project folder before continuing. Current value: {REPO_DIR}"
    )

os.chdir(REPO_DIR)
print(f"Using repo: {REPO_DIR}")

In [ ]:
import sys
from pprint import pprint

SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from colab_utils import download_dataset_snapshot, resolve_split_parquets

WORK_DIR = Path("/content/hateful_memes_colab")
DATA_DIR = WORK_DIR / "data"
CHECKPOINT_DIR = WORK_DIR / "checkpoints"
RESULTS_DIR = WORK_DIR / "results"

for folder in [WORK_DIR, DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

DATA_SOURCE = "huggingface_raw"  # "huggingface_raw" or "local_processed"
DATASET_REPO = "emily49/hateful-memes"

if DATA_SOURCE == "huggingface_raw":
    dataset_root = download_dataset_snapshot(
        repo_id=DATASET_REPO,
        local_dir=DATA_DIR / "raw",
    )
elif DATA_SOURCE == "local_processed":
    dataset_root = REPO_DIR / "Data" / "processed"
else:
    raise ValueError(f"Unsupported DATA_SOURCE: {DATA_SOURCE}")

split_paths = resolve_split_parquets(dataset_root)
pprint({key: str(value) for key, value in split_paths.items()})

In [ ]:
import importlib
import torch

from colab_utils import apply_training_overrides

RUN_MODE = "dynamic_v2"  # or "static_v2"

# These defaults are a little safer for typical Colab GPUs than the local settings.
EXTRA_CONFIG = {
    "batch_size": 4 if torch.cuda.is_available() else 2,
    "grad_accum": 8 if torch.cuda.is_available() else 4,
    "num_epochs": 10,
    "num_workers": 2,
    "pin_memory": bool(torch.cuda.is_available()),
}

module_name = "train_dynamic_v2" if RUN_MODE == "dynamic_v2" else "train_static_v2"
checkpoint_name = (
    "best_model_dynamic_v2_colab.pt"
    if RUN_MODE == "dynamic_v2"
    else "best_model_static_v2_colab.pt"
)

train_module = importlib.import_module(module_name)
importlib.reload(train_module)

config = apply_training_overrides(
    train_module,
    train_parquet=split_paths["train"],
    val_parquet=split_paths["validation"],
    checkpoint_dir=CHECKPOINT_DIR,
    checkpoint_name=checkpoint_name,
    extra_overrides=EXTRA_CONFIG,
)

print(f"CUDA available: {torch.cuda.is_available()}")
pprint(config)

In [ ]:
train_module.main()

print("\nSaved checkpoints:")
for path in sorted(CHECKPOINT_DIR.glob("*")):
    print(path)